In [5]:
import os
import polars as pl
from tqdm.notebook import tqdm
from collections import defaultdict

In [6]:
data_path = '../data/meds_normalized/data/train/'
downstream_idx = pl.read_parquet('../resources/downstream_idx.parquet')

In [7]:
import os
import polars as pl
from collections import Counter


def extract_query_features(query):

    features = {}

    # total events
    features["n_events"] = query.height

    # unique event types
    if "type" in query.columns:
        type_counts = (
            query
            .group_by("type")
            .len()
        )

        for row in type_counts.iter_rows(named=True):
            features[f"type_{row['type']}"] = row["len"]

    return features

In [8]:
import os
import polars as pl
from tqdm import tqdm


records_los = []
records_mort = []
records_icu = []
records_mort_1yr = []


for i in tqdm(range(len(downstream_idx))):

    stay = downstream_idx[i]

    subject_id = stay["subject_id"][0]
    icustay_id = stay["icustay_id"][0]
    split = stay["split"][0]
    shard = stay["shard"][0]

    shard_df = pl.read_parquet(
        os.path.join(data_path, shard)
    )

    timeline = shard_df.filter(
        pl.col("subject_id") == subject_id
    )


    def build_record(start_idx, end_idx, label_col):

        query = timeline[start_idx:end_idx]

        features = extract_query_features(query)

        # metadata first
        record = {
            "subject_id": subject_id,
            "icustay_id": icustay_id,
            "split": split,
        }

        # add extracted features
        record.update(features)

        # add label
        record["label"] = stay[label_col][0]

        return record


    # Long LOS 7d: within24
    records_los.append(
        build_record(
            stay["w24_start_1024"][0],
            stay["w24_end_1024"][0],
            "y_los_7"
        )
    )


    # In-hospital mortality: within48
    records_mort.append(
        build_record(
            stay["w48_start_1024"][0],
            stay["w48_end_1024"][0],
            "y_mort"
        )
    )


    # ICU readmission: withinStay
    records_icu.append(
        build_record(
            stay["wStay_start_1024"][0],
            stay["wStay_end_1024"][0],
            "y_icu_readmit_30"
        )
    )


    # 1 year mortality: withinStay
    records_mort_1yr.append(
        build_record(
            stay["wStay_start_1024"][0],
            stay["wStay_end_1024"][0],
            "y_mort_1yr"
        )
    )


# Convert to DataFrames
los_df = pl.DataFrame(records_los)
mort_df = pl.DataFrame(records_mort)
icu_df = pl.DataFrame(records_icu)
mort_1yr_df = pl.DataFrame(records_mort_1yr)


save_path = "./tabular_baselines"

os.makedirs(save_path, exist_ok=True)

los_df.write_parquet(
    os.path.join(save_path, "los_7.parquet")
)

mort_df.write_parquet(
    os.path.join(save_path, "mortality.parquet")
)

icu_df.write_parquet(
    os.path.join(save_path, "icu_readmit_30.parquet")
)

mort_1yr_df.write_parquet(
    os.path.join(save_path, "mortality_1yr.parquet")
)

  0%|          | 102/61175 [02:08<21:18:39,  1.26s/it]


KeyboardInterrupt: 

In [2]:
# y_los_7
# w24_start_1024
# w24_end_1024


# y_mort
# w48_start_1024
# w48_end_1024

# y_icu_readmit_30
# y_mort_1yr
# wStay_start_1024
# wStay_end_1024

In [3]:
from collections import defaultdict
import os
import polars as pl
from tqdm.notebook import tqdm

data_path = '/scratch/sas10092/ehr-foundation/data/meds_normalized/data/train/'
downstream_idx = pl.read_parquet('/scratch/sas10092/ehr-foundation/resources/downstream_idx.parquet')

In [4]:
downstream_idx

subject_id,hadm_id,hosp_admission_time,hosp_discharge_time,icustay_id,icu_admission_time,icu_discharge_time,in_hosp_mort_time,out_mortality_time,n_events_hosp,n_events_icu,shard,hosp_los,hosp_los_hours,hosp_los_days,icu_los,icu_los_hours,icu_los_days,mort_24hr_offset,mort_48hr_offset,y_mort,y_mort_1yr,y_mort_9mo,y_mort_6mo,y_mort_3mo,y_los_7,y_los_15,y_los_30,y_icu_readmit,y_icu_readmit_7,y_icu_readmit_15,y_icu_readmit_30,split,w24_min,w24_max,w48_min,w48_max,wStay_min,wStay_max,w24_start_512,w24_end_512,w24_start_1024,w24_end_1024,w24_start_1536,w24_end_1536,w24_start_2048,w24_end_2048,w48_start_512,w48_end_512,w48_start_1024,w48_end_1024,w48_start_1536,w48_end_1536,w48_start_2048,w48_end_2048,wStay_start_512,wStay_end_512,wStay_start_1024,wStay_end_1024,wStay_start_1536,wStay_end_1536,wStay_start_2048,wStay_end_2048
i64,i64,datetime[μs],datetime[μs],i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],u32,u32,str,duration[μs],f64,f64,duration[μs],f64,f64,datetime[μs],datetime[μs],i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
10000690,25860671,2150-11-02 18:02:00,2150-11-12 13:45:00,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,null,2152-01-30 00:00:00,4339,3981,"""40.parquet""",9d 19h 43m,235.716667,9.821528,3d 21h 26m 17s,93.438056,3.893252,2150-11-03 19:37:00,2150-11-04 19:37:00,0,0,0,0,0,0,0,0,0,0,0,0,"""train""",652,1797,652,2771,652,4788,1286,1797,774,1797,652,1797,652,1797,2260,2771,1748,2771,1236,2771,724,2771,4277,4788,3765,4788,3253,4788,2741,4788
10001217,24597018,2157-11-18 22:56:00,2157-11-25 18:00:00,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,null,null,1663,1418,"""2.parquet""",6d 19h 4m,163.066667,6.794444,1d 2h 49m 58s,26.832778,1.118032,2157-11-21 19:18:02,2157-11-22 19:18:02,0,0,0,0,0,0,0,0,0,0,0,0,"""train""",127,1382,127,1573,127,1573,871,1382,359,1382,127,1382,127,1382,1062,1573,550,1573,127,1573,127,1573,1062,1573,550,1573,127,1573,127,1573
10001725,25563031,2110-04-11 15:08:00,2110-04-14 15:00:00,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,null,null,1328,1126,"""337.parquet""",2d 23h 52m,71.866667,2.994444,1d 8h 7m 34s,32.126111,1.338588,2110-04-12 15:52:22,2110-04-13 15:52:22,0,0,0,0,0,0,0,0,0,0,0,0,"""test""",29,1005,29,1242,29,1242,494,1005,29,1005,29,1005,29,1005,731,1242,219,1242,29,1242,29,1242,731,1242,219,1242,29,1242,29,1242
10001884,26184834,2131-01-07 20:39:00,2131-01-20 05:15:00,37510196,2131-01-11 04:20:05,2131-01-20 08:27:30,2131-01-20 05:15:00,null,15837,14951,"""59.parquet""",12d 8h 36m,296.6,12.358333,9d 4h 7m 25s,220.123611,9.171817,2131-01-12 04:20:05,2131-01-13 04:20:05,1,0,0,0,0,1,0,0,0,0,0,0,"""train""",2961,5169,2961,6642,2961,18559,4658,5169,4146,5169,3634,5169,3122,5169,6131,6642,5619,6642,5107,6642,4595,6642,18048,18559,17536,18559,17024,18559,16512,18559
10002013,23581541,2160-05-18 07:45:00,2160-05-23 13:30:00,39060235,2160-05-18 10:00:53,2160-05-19 17:33:33,null,null,2186,1866,"""333.parquet""",5d 5h 45m,125.75,5.239583,1d 7h 32m 40s,31.544444,1.314352,2160-05-19 10:00:53,2160-05-20 10:00:53,0,0,0,0,0,0,0,0,0,0,0,0,"""train""",787,2617,787,2834,787,2834,2106,2617,1594,2617,1082,2617,787,2617,2323,2834,1811,2834,1299,2834,787,2834,2323,2834,1811,2834,1299,2834,787,2834
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
19999297,21439025,2162-08-14 23:55:00,2162-08-23 04:16:00,37364566,2162-08-16 05:48:32,2162-08-23 06:22:41,2162-08-23 04:16:00,null,10303,9669,"""30.parquet""",8d 4h 21m,196.35,8.18125,7d 34m 9s,168.569167,7.023715,2162-08-17 05:48:32,2162-08-18 05:48:32,1,0,0,0,0,1,0,0,0,0,0,0,"""train""",201,1757,201,3069,201,10387,1246,1757,734,1757,222,1757,201,1757,2558,3069,2046,3069,1534,3069,1022,3069,9876,10387,9364,10387,8852,10387,8340,10387
19999442,26785317,2148-11-19 10:00:00,2148-12-04 16:25:00,32336619,2148-11-19 14:23:43,2148-11-26 13:12:15,null,null,7521,707